# Week 6 · Random Variables, Sampling Distributions, and the Central Limit Theorem

*DS 207 · Week 6*

**Assignment Notebook**

**Author:** EC Corro, built in collaboration with Claude, ChatGPT, and Gemini

**Business scenario.** Visayas Fresh Logistics is a Cebu-based refrigerated delivery company serving supermarkets, hospitals, and restaurants under contracts that require at least 95% on-time delivery and box temperatures of 5°C or below. This assignment focuses on the provincial network, the North Cebu and South Cebu routes (3,000 completed deliveries in the ledger), where transit times are longer, more variable, and more strongly skewed than on the urban routes studied in the lecture notes. A late delivery carries a penalty of ₱0 if on time, ₱200 if up to 30 minutes late, and ₱500 if more than 30 minutes late.

**Submission.** Name your file `ds207_06_sampling_clt_lastname.ipynb`, replacing `lastname` with your surname, and upload it to your shared Google Drive folder by 11:59 PM on the day after the associated session, unless announced otherwise.

**AI use policy.** Generative AI is allowed for concept clarification, pseudocode, debugging, and idea generation. You must be able to explain any answer you submit without AI assistance, and you must cite any AI use.

**Points.** Section 1 (30 pts) + Section 2 (30 pts) = 60 pts.

## Learning Objectives

- Model delivery outcomes as discrete or continuous random variables and compute their expectation and variance.
- Derive and verify the sampling distribution of a sample mean, including its mean and standard error.
- Apply the Central Limit Theorem to a skewed variable, and judge when the normal approximation is and is not reliable.
- Turn probability calculations into a clear recommendation for an operational or contractual decision.

## Section 1: Concepts, Computation & Coding

This section is worth 30 points, 10 for each item. Run the cell below once before starting. It loads the provincial ledger into the DataFrame `deliveries`, with columns `route`, `delivery_min`, `promised_min`, `temp_c`, `late`, and `penalty_php`. Use `np.random.default_rng(207)` for every simulation so your results can be compared.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

import numpy as np
import pandas as pd

# Delivery ledger for Visayas Fresh Logistics: one row per completed cold-chain delivery.
rng = np.random.default_rng(207)

routes = {
    # route: (deliveries, median transit minutes, log-scale sigma, promised window, mean box temp, sd box temp)
    "Metro Cebu":  (3200, 26, 0.46, 75, 3.0, 0.8),
    "Mactan":      (1800, 30, 0.50, 90, 3.1, 0.8),
    "North Cebu":  (1600, 40, 0.58, 120, 3.4, 0.9),
    "South Cebu":  (1400, 38, 0.56, 120, 3.3, 0.9),
}

frames = []
for name, (n, med, sig, promised, t_mu, t_sd) in routes.items():
    minutes = 15 + rng.lognormal(mean=np.log(med), sigma=sig, size=n)
    frames.append(pd.DataFrame({
        "route": name,
        "delivery_min": minutes.round(1),
        "promised_min": promised,
        "temp_c": rng.normal(t_mu, t_sd, size=n).round(2),
    }))

ledger = pd.concat(frames, ignore_index=True)
ledger.insert(0, "delivery_id", np.arange(1, len(ledger) + 1))
delay = ledger["delivery_min"] - ledger["promised_min"]
ledger["late"] = (delay > 0).astype(int)
ledger["penalty_php"] = np.select([delay > 30, delay > 0], [500, 200], default=0)

# Assignment window: the provincial network (North Cebu and South Cebu routes)
deliveries = ledger[ledger["route"].isin(["North Cebu", "South Cebu"])].reset_index(drop=True)
deliveries.head()

### Item 1 · Penalties as a random variable (10 pts)

On the provincial routes, the penalty $X$ (in pesos) charged on a delivery is a discrete random variable. Rounded to three decimals, the ledger gives:

| Penalty $x$ | ₱0 | ₱200 | ₱500 |
|---|---|---|---|
| $p(x)$ | 0.957 | 0.027 | 0.016 |

Each week a provincial hub audits 25 deliveries and counts how many were late.

**(a) Concept (3 pts).** Classify transit minutes and the weekly late count as discrete or continuous. State the conditions under which the late count is a binomial random variable, and describe one realistic way the independence condition could fail on the provincial routes.

**(b) By hand (4 pts).** Using the table above, compute $E[X]$, $\operatorname{Var}(X)$, and $\operatorname{SD}(X)$. Then, treating the 25 audited deliveries as independent, find the expected value and standard deviation of the total penalty $T$ for one audit. Show your work.

**(c) Coding (3 pts).** From the full provincial ledger, compute the empirical probability table for `penalty_php` and its exact mean and variance, and compare them with your answers to (b). Then simulate 10,000 audits of 25 deliveries, and report the mean and standard deviation of the audit total.

**Answer (a)**

**Answer (b)**

In [ ]:
# TODO

### Item 2 · The sampling distribution of an audit average (10 pts)

A small feeder route has only three regular deliveries, with transit times of 40, 55, and 100 minutes. An auditor draws two deliveries at random with replacement and records their average transit time $\bar{X}$.

**(a) Concept (3 pts).** Explain the difference between the distribution of a single delivery time and the sampling distribution of the audit average. Say what stays the same between them, what changes, and why.

**(b) By hand (4 pts).** List all nine equally likely ordered samples and their means, and build the sampling distribution table of $\bar{X}$. Compute $E[\bar{X}]$ and $\operatorname{Var}(\bar{X})$ directly from the table, and verify them against $\mu$ and $\sigma^2/n$ for the three-delivery population.

**(c) Coding (3 pts).** Using `delivery_min` from the full provincial ledger, compute the population mean $\mu$ and population standard deviation $\sigma$ (use `ddof=0`). Simulate 10,000 audits of $n = 30$ deliveries, report the mean and standard deviation of the audit averages, compare them with $\mu$ and $\sigma/\sqrt{30}$, and report whether the audit averages are less skewed than individual delivery times.

**Answer (a)**

**Answer (b)**

In [ ]:
# TODO

### Item 3 · The Central Limit Theorem and audit size (10 pts)

A provincial hub audits $n = 36$ deliveries each week. For this network, single delivery times have mean $\mu = 61.2$ minutes, standard deviation $\sigma = 29.4$ minutes, and skewness of about 2.3.

**(a) Concept (3 pts).** Explain why the Central Limit Theorem allows a normal-distribution calculation for the weekly audit average even though single delivery times are strongly skewed. State one thing the theorem does not guarantee at $n = 36$, and explain how the skewness of the data affects the sample size needed.

**(b) By hand (4 pts).** Using the normal approximation and a standard normal table, compute (i) the standard error of the audit average, (ii) $P(\bar{X} > 70)$, (iii) the interval centered on $\mu$ that contains 95% of audit averages, and (iv) the smallest audit size $n$ for which the 95% margin of error for the mean is at most 5 minutes.

**(c) Coding (3 pts).** Simulate 20,000 audits of $n = 36$ deliveries from the provincial ledger. Compare the simulated $P(\bar{X} > 70)$ with your normal answer, report the share of simulated audit averages inside your interval from (b)(iii), and report the skewness of the audit averages. Then simulate 10,000 audits at your sample size from (b)(iv) and report the share of audit averages within 5 minutes of $\mu$.

**Answer (a)**

**Answer (b)**

In [ ]:
# TODO

## Section 2: Business Context & Reasoning (30 pts)

Each item below presents a decision facing a stakeholder at Visayas Fresh Logistics. Use the provincial ledger statistics from Section 1, state your assumptions, support your answer with a calculation, and end with a recommendation. Both technical soundness and business reasoning are graded, so a correct calculation without a usable recommendation, or a sensible recommendation without a correct calculation, will not earn full marks.

### Item 4 · An alarming audit at the provincial hub (15 pts)

This week's audit at the Provincial Hub sampled 36 deliveries and found 5 late (13.9%). The Provincial Hub Manager must decide whether to escalate to the Operations Director and pull trucks off the route for inspection. The ledger shows a long-run late rate of 4.3% on provincial routes, and the client contracts allow at most 5%.

Write your response in the cells below. It should:

1. State the assumptions you make about how the audit was drawn and which late rate you use as the benchmark.
2. Choose and justify a probability model, compute the probability of seeing 5 or more late deliveries in an audit of 36 if the true late rate is 4.3%, and comment on whether a normal approximation would be reliable here.
3. Interpret the result and recommend what the Hub Manager should do, including any follow-up action.

**Point split:** technical soundness (8 pts): assumptions 2, model and computation 3, interpretation 3; business reasoning (7 pts): recommendation follows from the result 4, practical follow-up and risk awareness 3.

**Assumptions and technical analysis**

In [ ]:
# TODO

**Interpretation and recommendation**

### Item 5 · A proposed contract clause (15 pts)

A supermarket chain proposes this clause for provincial deliveries: if the average transit time in the weekly audit of 36 deliveries exceeds 65 minutes, Visayas Fresh Logistics pays a ₱20,000 penalty for that week. The Commercial Director asks whether to accept. Use $\mu = 61.2$ minutes and $\sigma = 29.4$ minutes for provincial single-delivery times.

Write your response in the cells below. It should:

1. State the assumptions you make, including why a normal calculation is or is not defensible here.
2. Compute the probability that a given week triggers the penalty by chance alone, the expected number of triggered weeks and expected penalty cost over 52 weeks, and the same figures if the audit were 100 deliveries.
3. Recommend whether to accept, reject, or counter-propose, with specific terms and the reasoning behind them.

**Point split:** technical soundness (8 pts): assumptions 2, computation 3, interpretation 3; business reasoning (7 pts): recommendation follows from the result 4, negotiation practicality and risk awareness 3.

**Assumptions and technical analysis**

In [ ]:
# TODO

**Interpretation and recommendation**

**Citation reminder.** All external sources, including textbooks, papers, articles, code snippets, and any use of AI tools, must be cited in accordance with course policy. Answers that rely on uncited external material will be marked down.

## References

Blitzstein, J. K., & Hwang, J. (2019). *Introduction to probability* (2nd ed.). CRC Press.

Casella, G., & Berger, R. L. (2002). *Statistical inference* (2nd ed.). Duxbury.

Ross, S. M. (2019). *A first course in probability* (10th ed.). Pearson.

Wasserman, L. (2004). *All of statistics: A concise course in statistical inference*. Springer.